# registry

> Defining hyperelements and collecting their styles and behaviors

In [ ]:
#| default_exp registry

AnswerUI provides built-in components through its shared stylesheet and JavaScript assets. Python constructors render their marked HTML. This registry holds application CSS and event handlers, attaches custom rules to constructors, and assembles those additions for each page. Built-in CSS remains in the shared assets instead of being copied into registry entries.

In [ ]:
import hashlib, inspect, json
from collections.abc import Mapping
from fastcore.utils import *
import fasthtml.components
from answerui import assets as shared_assets

In [ ]:
from fasthtml.common import Label, Input, Button, to_xml
from fastcore.test import *

## Rules

Both styling and behavior pair a CSS selector with an action: CSS declarations for styling, or an event name and JavaScript handler for behavior. A `Rule` stores both CSS text and behavior records under a [cascade layer](https://developer.mozilla.org/en-US/docs/Web/CSS/@layer) because each page loads one combined stylesheet and one combined script.

The layer list is fixed and ordered. For normal declarations, each layer overrides earlier layers regardless of specificity. You can restyle a built-in component from the `app` layer without changing its definition. Component and base CSS do not use `!important` because `!important` reverses cascade-layer order (only the hidden-state classes in `utilities` use it).


In [ ]:
LAYERS = ('reset','tokens','base','layout','components','app','utilities')

Behaviors are passed as a dict from `(selector, event)` keys to JavaScript handler bodies. Each handler runs with `el` bound to the matched element and `e` to the event. For an event that doesn't bubble, such as `toggle`, add a truthy third item to the key to attach the handler in the capture phase. When a rule has a root selector `sel`, `_behaviors` replaces each `&` in its behavior selectors with `:is(sel)` (raising `ValueError` if `sel` is missing).

In [ ]:
def _behaviors(on, sel):
    "Behavior records with each `&` in a selector replaced by `:is(sel)`, to keep a selector list grouped"
    res = []
    for (selector, event, *capture), js in (on or {}).items():
        if '&' in selector:
            if sel is None: raise ValueError(f'`&` in {selector!r} needs sel')
            selector = selector.replace('&', f':is({sel})')
        res.append(dict(sel=selector, ev=event, js=js, cap=bool(capture and capture[0])))
    return res

A rule's key is a name, or `owner.name` when the rule belongs to a hyperelement. `Rule` checks the layer, expands the behaviors, and derives `owner` and `name` from the key. For a variant rule, the variant attribute is stored in `variant`:


In [ ]:
class Rule:
    "A named CSS and behavior definition in a cascade layer"
    def __init__(
        self,
        key,          # Registry key: a name, or `owner.name` for a hyperelement's rule
        css='',       # Complete CSS, or a no-argument function returning it
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        if layer not in LAYERS: raise ValueError(f'Unknown layer: {layer}')
        store_attr('key,css,sel,layer,doc,variant')
        self.on = _behaviors(on, sel)

    @property
    def owner(self): return self.key.split('.', 1)[0] if '.' in self.key else None

    @property
    def name(self): return self.key.split('.', 1)[-1]

    @property
    def compiled_css(self):
        "Evaluate callable CSS and return the stylesheet text"
        return (self.css() if callable(self.css) else self.css).strip()

    def __repr__(self):
        sel = f' {self.sel}' if self.sel else ''
        doc = f': {self.doc}' if self.doc else ''
        return f'{self.key} [{self.layer}]{sel}{doc}'

`Registry.rule` takes CSS first and an optional name second. Without a name, it hashes the CSS, selector, expanded behaviors and layer into a deterministic key. Identical registrations share one entry, including behavior-only rules. Pass an explicit name when `css` is callable, or when later edits should replace the previous rule rather than add another. A component's rule uses `owner.name` as its key.

Rules are stored in `Registry.rules`. Registering an existing key updates that `Rule` object in place, preserving both existing references and its position in the stylesheet. Registering or removing a rule increments `version`, which `Registry.assets` checks before rebuilding the stylesheet and script. Each entry of `layers` is a live `RuleView` of the rules in that layer:


In [ ]:
class RuleView(Mapping):
    "A live, read-only view of one layer or one hyperelement's rules"
    def __init__(self, registry, *, layer=None, owner=None): store_attr('registry,layer,owner')
    def _matches(self, r): return (self.layer is None or r.layer==self.layer) and (self.owner is None or r.owner==self.owner)
    def __iter__(self): return (r.name if self.owner is not None else k for k,r in self.registry.rules.items() if self._matches(r))
    def __len__(self): return sum(1 for _ in self)
    def __getitem__(self, name):
        key = f'{self.owner}.{name}' if self.owner is not None else name
        r = self.registry.rules[key]
        if not self._matches(r): raise KeyError(name)
        return r
    def __repr__(self): return '\n'.join(repr(r) for r in self.values()) or '(no rules)'

class Registry:
    "Named rules and hyperelements, from which a page's stylesheet and script are built"
    def __init__(self):
        self.rules, self.elements, self.files, self.version = {}, {}, {}, 0
        self.layers = {name: RuleView(self, layer=name) for name in LAYERS}

    def rule(
        self,
        css='',       # Complete CSS, or a no-argument function returning it
        name=None,    # Registry key; omit for a hash of the CSS, selector, behaviors and layer
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors; does not scope CSS
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        "Register CSS and behaviors, replacing an existing name in place"
        if name is None and not isinstance(css, str): raise ValueError('Callable CSS requires a name')
        r = Rule(name or '', css, on=on, sel=sel, layer=layer, doc=doc, variant=variant)
        if name is None: r.key = hashlib.sha256(json.dumps((css, sel, r.on, layer)).encode()).hexdigest()[:12]
        if r.key in self.rules: self.rules[r.key].__dict__.update(vars(r))
        else: self.rules[r.key] = r
        self.version += 1
        return self.rules[r.key]

    def remove(self, key):
        "Remove and return the rule at its full registry key"
        r = self.rules.pop(key)
        self.version += 1
        return r

    def file(
        self,
        name, # Name the file is served under
        path, # Where the file is on disk
        url,  # A public URL of the same file, which rules use to refer to it
    ):
        "Register a file that rules refer to by `url`, and return `url`. `Registry.serve` serves a copy of the file at `path` and replaces `url` with that copy's URL in the stylesheet and script"
        self.files[name] = (Path(path), url)
        return url

    def __repr__(self): return f'Registry({len(self.rules)} rules, {len(self.elements)} elements)'

    def _repr_markdown_(self):
        layers = '\n'.join(f'- `{name}`: {len(rules)} rules' for name,rules in self.layers.items())
        elements = ', '.join(f'`{k}`' for k in self.elements) or 'none'
        rules = '\n'.join(f'- `{k}` ({r.layer})' + (f': {r.doc}' if r.doc else '') for k,r in self.rules.items() if r.owner is None) or 'none'
        return f'{self!r}\n\nLayers:\n{layers}\n\nElements: {elements}\n\nRules of no element:\n{rules}'

reg = Registry()

The simplest rule is bare CSS with no constructor, such as a centered page column. Its layer defaults to `app`. Registering the same name again returns the existing entry:


In [ ]:
page_css = 'main { max-width: 65ch; margin-inline: auto; padding: 1rem; }'
page_rule = reg.rule(page_css, 'app-page', doc='A centered single-column page.')
test_eq(page_rule.css, page_css)
test_is(reg.rule(page_css, 'app-page', doc='A centered single-column page.'), page_rule)
page_rule

app-page [app]: A centered single-column page.

A standalone behavior works the same way. Here we add a `select-all` attribute that selects an input's text on focus. Passing `sel='input[select-all]'` sets the root selector that `&` expands to in the handler key. The rule stores the expanded behavior record:

In [ ]:
select_all = reg.rule(name='select-all', sel='input[select-all]', on={('&', 'focusin'): r"el.select()"},
    doc='Select the input text when it receives focus.')
select_all.on

[{'sel': ':is(input[select-all])',
  'ev': 'focusin',
  'js': 'el.select()',
  'cap': False}]

In [ ]:
with expect_fail(ValueError): reg.rule('a { color: teal; }', layer='appp')
with expect_fail(ValueError): reg.rule(name='paste', on={('&', 'click'): ''})

Displaying `reg` lists each rule that belongs to no component, such as `app-page` and `select-all`:

In [ ]:
reg

Registry(2 rules, 0 elements)

Layers:
- `reset`: 0 rules
- `tokens`: 0 rules
- `base`: 0 rules
- `layout`: 0 rules
- `components`: 0 rules
- `app`: 2 rules
- `utilities`: 0 rules

Elements: none

Rules of no element:
- `app-page` (app): A centered single-column page.
- `select-all` (app): Select the input text when it receives focus.

## Building the stylesheet

`build_css` starts with the shared `styles.css`, whose opening `@layer` statement declares the layer order. Registered rules follow as one `@layer` block per layer in registration order, each preceded by a comment naming its key. Application rules default to `app`, after the built-in `components` layer. For CSS that depends on later registrations, pass a zero-argument callable that `build_css` evaluates during assembly.


In [ ]:
@patch
def build_css(self:Registry):
    "The shared foundation followed by registered CSS in cascade-layer order"
    out = [(shared_assets/'styles.css').read_text()]
    for layer, rules in self.layers.items():
        parts = [f'/* {r.key} */\n{css}' for r in rules.values() if (css := r.compiled_css)]
        if parts: out.append(f'@layer {layer} {{\n' + '\n'.join(parts) + '\n}')
    return '\n\n'.join(out) + '\n'

In [ ]:
reg.layers['app']

app-page [app]: A centered single-column page.
select-all [app] input[select-all]: Select the input text when it receives focus.

## Hyperelements

A component pairs a constructor (a standard FastTag or a Python function combining multiple tags) with registered rules. Calling a `Hyperelement` invokes that constructor. Its `sel` attribute holds the root selector. Its `rules` attribute is a live `RuleView` of the component's rules:


In [ ]:
def _nest(sel, css):
    if callable(css): return lambda: _nest(sel, css())
    return f'{sel} {{\n{css.strip()}\n}}' if css.strip() else ''
def _generic(*children, **attrs): ...

class Hyperelement:
    "A callable constructor linked to its registered rules and behaviors"
    def __init__(self, registry, name, fn, sel, doc='', generated=False):
        self.registry, self.__name__, self.fn, self.sel = registry, name, fn, sel
        self.__doc__, self.generated = doc, generated
        self.rules = RuleView(registry, owner=name)

    def __call__(self, *c, **attrs): return self.fn(*c, **attrs)

    @property
    def __signature__(self): return inspect.signature(_generic if self.generated else self.fn)

    @property
    def variants(self):
        "Declared variant attributes and their rules"
        return {r.variant: r for r in self.rules.values() if r.variant is not None}

    @property
    def behaviors(self):
        "Behavior records declared by this hyperelement's rules"
        return [b for r in self.rules.values() for b in r.on]

    def __repr__(self): return f'{self.__name__} [{self.sel}]'

    def _repr_markdown_(self):
        name = self.__name__
        sig = str(self.__signature__)
        parts = [f'### `{name}{sig}`', self.__doc__, f'Root selector: `{self.sel}`.']
        if self.variants:
            variants = [f'- `{attr}`: {r.doc}' for attr,r in self.variants.items()]
            parts.append('Declared variants:\n' + '\n'.join(variants))
        rules = [f'- `{k}` ({r.layer})' + (f': {r.doc}' if r.doc and r.doc != self.__doc__ else '') for k,r in self.rules.items() if r.variant is None]
        parts.append('Rules:\n' + '\n'.join(rules))
        return '\n\n'.join(p for p in parts if p)

Use `Hyperelement.rule` to add CSS nested under the root selector, or `Hyperelement.variant` to add an attribute variant nested under `:is(sel)[name]`. Both register their rule with the hyperelement as its owner. Calling `variant` also stores the attribute name on the rule (`base` is reserved for the constructor's own rule):

In [ ]:
@patch
def rule(
    self:Hyperelement,
    name,         # Local rule name; registering it again replaces that rule
    css='',       # CSS declarations and nested rules, scoped to the root selector
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the root selector
    layer='app',  # One of LAYERS
    doc='',       # Description of the customization
):
    "Register CSS and behaviors scoped to this hyperelement"
    return self.registry.rule(_nest(self.sel, css), f'{self.__name__}.{name}', on=on, sel=self.sel, layer=layer, doc=doc)

@patch
def variant(
    self:Hyperelement,
    name,         # Attribute selecting the variant; also its local rule name
    css='',       # CSS declarations and nested rules, scoped to the variant
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the variant selector
    layer='app',  # One of LAYERS
    doc='',       # Description shown with the variant
):
    "Declare an attribute variant without changing the constructor"
    if name=='base': raise ValueError('base is reserved for the constructor definition')
    sel = f':is({self.sel})[{name}]'
    return self.registry.rule(_nest(sel, css), f'{self.__name__}.{name}', on=on, sel=sel, layer=layer, doc=doc, variant=name)

`hypelt` registers a component from a FastHTML tag name or a handwritten constructor and nests `css` under its root selector as the `base` rule, just like `Hyperelement.rule`. Write declarations for the root, nested selectors for children, and `&` for states or pseudo-elements on the root. Pass a callable with no parameters when `css` depends on later registrations. Put global rules such as `@keyframes` in `reg.rule`, which accepts complete CSS without wrapping and uses its `sel` parameter for behaviors only. The module-level `hypelt` is bound to `reg`:


In [ ]:
@patch
def hypelt(
    self:Registry,
    name=None,   # FastHTML tag name or a constructor; omit to use as a decorator
    *,
    sel=None,    # Root selector; required for a handwritten constructor
    css='',      # CSS declarations and nested rules, or a no-argument function returning them
    on=None,     # Mapping of (selector, event[, capture]) to JavaScript handler bodies
    layer='app', # One of LAYERS; built-ins use components, layout, or base
    doc=None,    # Description; defaults to a handwritten constructor's docstring
):
    "Register a constructor and its base rule, returning a callable hyperelement"
    if name is None: return partial(self.hypelt, sel=sel, css=css, on=on, layer=layer, doc=doc)
    generated = isinstance(name, str)
    if generated:
        fn = getattr(fasthtml.components, name)
        if sel is None: sel = fn().tag
    else:
        fn, name = name, name.__name__
        if sel is None: raise ValueError('A handwritten constructor needs sel')
        if doc is None: doc = fn.__doc__
    if name in self.elements and self.elements[name].sel != sel:
        raise ValueError(f'{name} is registered with root selector {self.elements[name].sel!r}, which its rules and variants target; register a new hyperelement for a new root')
    self.rule(_nest(sel, css), f'{name}.base', on=on, sel=sel, layer=layer, doc=doc or '')
    obj = Hyperelement(self, name, fn, sel, doc or '', generated)
    if name in self.elements: self.elements[name].__dict__.update(vars(obj))
    else: self.elements[name] = obj
    return self.elements[name]

hypelt = reg.hypelt

As an example component, let's build a dismissible notice box with a close button. Because HTML has no built-in notice element, we use the custom tag `notice-box`. Clicking any child with the `dismiss` attribute removes the notice:

In [ ]:
notice_css = r"""
display: block;
padding: .75rem 1rem;
background: #f4f4f6;
border: 1px solid #ccc;
border-radius: 8px;
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, on={('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"},
    layer='components', doc='A dismissible notice.')
to_xml(NoticeBox('Changes saved.', Button('Dismiss', dismiss=True)))

'<notice-box>Changes saved.<button dismiss>Dismiss</button></notice-box>'

For a string name such as `'NoticeBox'`, `hypelt` looks up the constructor on `fasthtml.components` and calls it with no arguments to get the root selector (`notice-box`). `NoticeBox.rules` and `reg.rules` hold the same `Rule` object:


In [ ]:
test_eq(NoticeBox().tag, NoticeBox.sel)
assert NoticeBox.rules['base'] is reg.rules['NoticeBox.base']
assert NoticeBox.rules['base'].compiled_css.startswith('notice-box {')
NoticeBox

### `NoticeBox(*children, **attrs)`

A dismissible notice.

Root selector: `notice-box`.

Rules:
- `base` (components)

The second case is a handwritten constructor, such as a labelled field that wraps an `input` inside its `label` to associate them without an `id`. Because `Field` requires a `label` argument, `hypelt` can't call it with no arguments to infer a root selector. Instead we mark the `label` with a `field` attribute and pass `sel='label[field]'` using the decorator form:

In [ ]:
field_css = r"""
display: grid;
gap: .5rem;
"""

@hypelt(sel='label[field]', css=field_css)
def Field(label, **attrs):
    "A labelled input"
    return Label(label, Input(**attrs), field=True)

to_xml(Field('Email', name='email', type='email'))

'<label field>Email  <input name="email" type="email">\n</label>'

## Customization

There are two ways to change a component's styling or behavior. To override a component without altering its `base` rule, register separate rules in the `app` layer. To redefine it, call `hypelt` again with the same name to replace its constructor and `base` rule in place.

### Overrides

Say a developer wants rounder notices in their application. `NoticeBox.rule` registers a named rule in the `app` layer by default, where it overrides the component's CSS in the `components` layer:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notices in this application.')


NoticeBox.corners [app] notice-box: Rounder notices in this application.

To select a style with an attribute, declare it with `variant`. Its description then appears in the hyperelement's documentation while the existing constructor accepts the attribute:


In [ ]:
NoticeBox.variant('quiet', 'background: transparent; border-color: transparent;', doc='An unfilled notice without a visible border.')


NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.

In [ ]:
test_eq(list(NoticeBox.variants), ['quiet'])
to_xml(NoticeBox('Saved quietly.', quiet=True))

'<notice-box quiet>Saved quietly.</notice-box>'

To change a component only within part of a page, register an application rule scoped to that container. Here `reg.rule` registers `editor-notices` to square the corners of every `notice-box` inside an `[editor]` element, with an explicit name for later replacement. Because this rule is not registered on `NoticeBox`, it leaves the component's definition untouched and appears in `reg.layers['app']` rather than `NoticeBox.rules`:


In [ ]:
reg.rule('[editor] notice-box { border-radius: 0; }', 'editor-notices', doc='Square notices inside the editor.')
reg.layers['app']

app-page [app]: A centered single-column page.
select-all [app] input[select-all]: Select the input text when it receives focus.
Field.base [app] label[field]: A labelled input
NoticeBox.corners [app] notice-box: Rounder notices in this application.
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.

### Redefinitions

Calling `hypelt` again with the same name updates the hyperelement's constructor and `base` rule in place, preserving existing references, application rules, and declared variants. Omitted `css` or `on` arguments are cleared rather than merged because the new call replaces the entire `base` rule. `hypelt` raises `ValueError` if `sel` changes because existing rules and variants are nested under the original root selector. Omitting `on` here makes `NoticeBox` persistent:


In [ ]:
original, base = NoticeBox, NoticeBox.rules['base']
NoticeBox = hypelt('NoticeBox', css=lambda: notice_css.replace('8px', '12px'), layer='components', doc='A persistent notice.')
test_is(NoticeBox, original)
test_is(NoticeBox.rules['base'], base)
assert '12px' in base.compiled_css
assert 'corners' in NoticeBox.rules and 'quiet' in NoticeBox.variants
with expect_fail(ValueError): hypelt('NoticeBox', sel='notice-box[persistent]', css=notice_css, layer='components')
NoticeBox

### `NoticeBox(*children, **attrs)`

A persistent notice.

Root selector: `notice-box`.

Declared variants:
- `quiet`: An unfilled notice without a visible border.

Rules:
- `base` (components)
- `corners` (app): Rounder notices in this application.

The base rule now has no behaviors. To remove the application override, pass its full key to `reg.remove`. A layer view taken earlier no longer lists it:


In [ ]:
test_eq(NoticeBox.behaviors, [])
app_rules = reg.layers['app']
v = reg.version
reg.remove('NoticeBox.corners')

NoticeBox.corners [app] notice-box: Rounder notices in this application.

In [ ]:
assert 'NoticeBox.corners' not in app_rules
test_eq(reg.version, v+1)
app_rules

app-page [app]: A centered single-column page.
select-all [app] input[select-all]: Select the input text when it receives focus.
Field.base [app] label[field]: A labelled input
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.